# FairyZero — trần hiệu năng thật sự của T4  *(bản 3)*

**Đổi lớn so với bản 2.** Phần đo quyết định giờ chạy bằng **chính engine**,
không qua Python nữa.

Bản 2 hỏng vì: Colab giờ dùng **Python 3.13**, mà ONNX Runtime **1.20.1**
(bản engine dùng) không có wheel cho 3.13 → pip rơi về **1.30.0** → bản đó đòi
**CUDA 13** trong khi torch của Colab mang **CUDA 12** → `libcublasLt.so.13:
cannot open shared object file`.

Engine thì **mang theo ORT 1.20.1 của riêng nó** trong `third_party/`. Nên đo
bằng engine vừa thoát hẳn mớ phụ thuộc pip, vừa **trung thực hơn**: đúng ORT,
đúng session options, đúng buffer mà self-play dùng.

| Phần | Đo gì | Thời gian |
|---|---|---|
| Build | dựng engine | ~10 phút |
| **A** | `--bench-nn`: suy luận thuần tuý, không MCTS | **~3 phút** |
| A2 | Python + TensorRT (best-effort) | 15-25 phút |
| **C** | quét `ev/play` | ~25 phút |

**Nếu ít thời gian: chạy tới hết Phần A rồi gửi tôi.** Riêng nó đã phân được
ba nhánh quyết định.


## 0. Chuẩn bị + build engine


In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv
%cd /content
!rm -rf chess_variant_engine
!git clone -q --depth 1 -b mcts-capacity-256 https://github.com/phuc11731510/chess_variant_engine.git
!wget -q -nc https://github.com/phuc11731510/chess_variant_engine/releases/download/v1.0.0/12bx144fx8s_12.onnx
!ls -la 12bx144fx8s_12.onnx


In [ ]:
!bash /content/chess_variant_engine/custom_engine/scripts/colab_setup.sh 2>&1 | tail -5
!bash /content/chess_variant_engine/custom_engine/scripts/colab_prebuilt.sh wrap


---
# PHẦN A — Suy luận thuần tuý, bằng chính engine  ⬅ QUAN TRỌNG NHẤT

Không MCTS, không cây, không cache. Chỉ nạp mạng rồi gọi `ComputeBlocking()`
lặp đi lặp lại, quét qua nhiều cỡ batch.

In **hai** bảng:
- **DYNAMIC batch** — mỗi `Run()` nhận đúng batch đưa vào, không pad.
- **FIXED batch = 16** — đúng đường self-play thật đang chạy, có pad.

Chênh lệch giữa hai bảng chính là chi phí padding, hiện ra bằng số.


In [ ]:
!bash /content/chess_variant_engine/custom_engine/run.sh --bench-nn \
    --weights /content/12bx144fx8s_12.onnx \
    --provider cuda --fixed-batch 16


### Đọc bảng thế nào

| Suy luận thuần (dynamic, batch 16-64) | Kết luận |
|---|---|
| ≈ **2.200 pos/giây** | T4 đúng là trần. Phần mềm hết đường. Chỉ còn fp16 hoặc phần cứng |
| **≫ 2.200** (2× trở lên) | Engine mất hiệu năng ở khâu **điều phối CPU/GPU** — nghi can là thiết kế *stop-the-world* trong `BatchingBackend`. Sửa được, miễn phí |

Cột `us/vi tri` là thời gian mỗi vị trí. Nếu nó **giảm** khi batch tăng thì
batch lớn có giá trị ở tầng suy luận; nếu **phẳng** thì GPU đã bão hoà tính toán.


---
# PHẦN A2 — TensorRT qua Python  *(best-effort)*

Engine chưa có nhánh TensorRT nên câu hỏi này vẫn phải hỏi qua Python.

⚠ Bản ORT ở đây sẽ **khác** bản engine dùng (1.30 vs 1.20.1). Nên con số tuyệt
đối **không so thẳng** với Phần A được — nhưng **tỉ số TensorRT/CUDA trong cùng
một bản ORT** thì vẫn trả lời đúng câu "TensorRT có đáng không".

Script sẽ tự cài bộ thư viện `nvidia-*-cu13` để vá lỗi `libcublasLt.so.13`.

TensorRT biên dịch kế hoạch riêng cho từng cỡ batch: 1-5 phút mỗi cỡ.


In [ ]:
!bash /content/chess_variant_engine/custom_engine/scripts/bench_ort.sh \
    /content/12bx144fx8s_12.onnx --board 10


### Nếu A2 vẫn báo `KHONG kich hoat duoc`

Không sao — **Phần A đã đủ để quyết định**. Chạy ô chẩn đoán rồi gửi tôi output,
còn lại cứ tiếp tục Phần C.


In [ ]:
import glob, os, site, subprocess
print("== goi onnxruntime dang cai ==")
subprocess.run("pip list 2>/dev/null | grep -i -E 'onnxruntime|nvidia-cu'", shell=True)
print()
print("== phien ban Python cua Colab ==")
subprocess.run("python3 --version", shell=True)
print()
print("== thu vien CUDA tim thay (theo so phien ban) ==")
for pat in ["libcublasLt.so*", "libcudnn.so*", "libcudart.so*"]:
    subprocess.run("find /usr/local/lib -name '%s' 2>/dev/null | head -4" % pat, shell=True)


---
# PHẦN C — Cắt `ev/play`

`ev/play = 1,43` nghĩa là cứ 1 playout hữu ích thì engine gửi 1,43 thế cờ lên
GPU — **30% công việc GPU không sinh ra gì**. Nguyên nhân là *collision*: khi
gom một lô, các lần đi xuống cây sau chưa biết kết quả của lần trước nên hay
rơi trúng cùng một lá chưa được đánh giá.

Cách cắt: **gom lô nhỏ hơn** → ít lần đi xuống mù hơn → ít va chạm hơn.

Phép đo lần trước đã chứng minh **batch to không mang lại throughput**, nên
giảm lô gần như miễn phí. 5 nhánh × 4 phút ≈ 25 phút.


In [ ]:
import subprocess, re, os

ENG  = "/content/chess_variant_engine/custom_engine/run.sh"
W    = "/content/12bx144fx8s_12.onnx"
SECS = 240

# minibatch-size=0 nghia la "dung goi y cua backend" = fixed-batch.
# Lo nho hon => it lan di xuong mu hon => it collision => ev/play thap hon.
ARMS = [
    ("C1_goc_mb16", ["--fixed-batch", "16"], []),
    ("C2_mb8",      ["--fixed-batch", "8"],  ["--search-opt", "minibatch-size=8"]),
    ("C3_mb4",      ["--fixed-batch", "4"],  ["--search-opt", "minibatch-size=4"]),
    ("C4_mb2",      ["--fixed-batch", "2"],  ["--search-opt", "minibatch-size=2"]),
    ("C5_mb8_col4", ["--fixed-batch", "8"],  ["--search-opt", "minibatch-size=8",
                                              "--search-opt", "max-collision-events=4"]),
]

rows = []
for name, fb, extra in ARMS:
    out = "/content/bench_c/" + name
    subprocess.run(["rm", "-rf", out])
    os.makedirs(out, exist_ok=True)
    cmd = ["bash", ENG, "--selfplay", "--games", "100000", "--max-seconds", str(SECS),
           "--visits", "800", "--max-moves", "400", "--temp-cutoff", "32",
           "--parallel", "4", "--provider", "cuda"] + fb + extra + \
          ["--noise-alpha", "0.15", "--show-nps", "--weights", W, "--out", out]
    print(">>> %s : %s" % (name, " ".join(fb + extra)))
    r = subprocess.run(cmd, capture_output=True, text=True)
    log = r.stdout + r.stderr
    def g(pat):
        m = re.search(pat, log)
        return m.group(1) if m else "-"
    row = (name,
           g(r"Finished (\d+)/"),
           g(r"Van/gio\s*:\s*([\d.]+)"),
           g(r"NN eval/giay\s*:\s*([\d.]+)"),
           g(r"NN eval/playout\s*:\s*([\d.]+)"),
           g(r"Batch TB moi Run\(\)\s*:\s*([\d.]+)"),
           g(r"Phi do pad\s*:\s*([\d.]+)"))
    print("    van=%s  van/gio=%s  eval/s=%s  ev/play=%s  batch=%s  pad=%s%%" % row[1:])
    rows.append(row)
    subprocess.run(["rm", "-rf", out])

print()
print("%-14s%5s%10s%10s%9s%9s%8s" % ("arm", "van", "van/gio", "eval/s", "ev/play", "batchTB", "pad%"))
print("-" * 65)
for r_ in rows:
    print("%-14s%5s%10s%10s%9s%9s%8s" % r_)


---
# Gửi lại cho tôi

1. **Hai bảng của Phần A** — ưu tiên số một
2. Bảng cuối **Phần C**
3. **A2** nếu chạy được; nếu không thì output ô chẩn đoán
